In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from openai import OpenAI
import os
import numpy as np

# Set your OpenAI API key as an environment variable
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Initialize the OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Load the tokenizer and model from the 'law-ai/InLegalBERT' repository
tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
model = AutoModel.from_pretrained("law-ai/InLegalBERT")

# Define the input text
text = "Draft a MOU between party A and Party B"

try:
    # Tokenize the input text and return PyTorch tensors
    encoded_input = tokenizer(text, return_tensors="pt")
    print("Tokenization successful")

    # Pass the tokenized input to the model
    with torch.no_grad():
        output = model(**encoded_input)
    print("Model output generated")

    # Extract the last hidden state from the model's output
    last_hidden_state = output.last_hidden_state
    print("Last hidden state extracted")

    # Calculate the mean of the last hidden state to get a single vector representation
    mean_state = torch.mean(last_hidden_state, dim=1).squeeze()
    print(f"Mean state shape: {mean_state.shape}, type: {type(mean_state)}")

    # Convert to numpy array explicitly
    text_embedding = mean_state.detach().cpu().numpy()
    print(f"Text embedding type: {type(text_embedding)}")
    print(f"Is text_embedding a numpy array? {isinstance(text_embedding, np.ndarray)}")
    
    if isinstance(text_embedding, np.ndarray):
        print(f"Text embedding shape: {text_embedding.shape}")
        print(f"Text embedding dtype: {text_embedding.dtype}")
        print(f"First few values of text_embedding: {text_embedding[:5]}")
    else:
        print("text_embedding is not a numpy array. Attempting to convert...")
        text_embedding = np.array(text_embedding)
        print(f"After conversion - Is text_embedding a numpy array? {isinstance(text_embedding, np.ndarray)}")
        if isinstance(text_embedding, np.ndarray):
            print(f"After conversion - Text embedding shape: {text_embedding.shape}")
            print(f"After conversion - Text embedding dtype: {text_embedding.dtype}")
            print(f"After conversion - First few values of text_embedding: {text_embedding[:5]}")

    # Convert the embedding to a string representation
    try:
        embedding_str = ','.join(map(str, text_embedding.flatten()))
        print("Embedding converted to string")
    except Exception as e:
        print(f"Error converting embedding to string: {str(e)}")
        raise

    # Prepare a more detailed prompt using the embedding information
    detailed_prompt = f"""
    Based on the legal context embedding: {embedding_str[:100]}... (truncated for brevity)

    Draft a Memorandum of Understanding (MOU) between Party A and Party B. Consider the following points:
    1. Purpose of the agreement
    2. Scope of cooperation
    3. Responsibilities of each party
    4. Duration of the agreement
    5. Confidentiality clauses
    6. Termination conditions

    Please provide a structured draft of the MOU.
    """
    print("Detailed prompt prepared")

    # Generate a draft using OpenAI's GPT model
    response = client.chat.completions.create(
        model="gpt-3.5-turbo-16k",
        messages=[
            {"role": "system", "content": "You are a legal expert assistant specializing in drafting legal documents. Use the provided legal context to inform your draft."},
            {"role": "user", "content": detailed_prompt}
        ],
        max_tokens=1000,
        temperature=0.7
    )
    print("OpenAI API call successful")

    # Extract the generated text
    draft = response.choices[0].message.content.strip()

    # Print the draft
    print("Generated Draft:")
    print(draft)

except Exception as e:
    print(f"An error occurred: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from openai import OpenAI
import os

# Set your OpenAI API key
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
client = OpenAI(api_key=OPENAI_API_KEY)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
model = AutoModel.from_pretrained("law-ai/InLegalBERT")

# Define input text
text = "Draft a MOU between party A and Party B"

# Tokenize and get model output
encoded_input = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    output = model(**encoded_input)

# Get text embedding
last_hidden_state = output.last_hidden_state
text_embedding = torch.mean(last_hidden_state, dim=1).squeeze().detach().cpu().numpy()

# Convert embedding to string
embedding_str = ','.join(map(str, text_embedding.flatten()))

# Prepare prompt
detailed_prompt = f"""
Based on the legal context embedding: {embedding_str}

Draft a Memorandum of Understanding (MOU) between Party A and Party B. Consider the following points:
1. Purpose of the agreement
2. Scope of cooperation
3. Responsibilities of each party
4. Duration of the agreement
5. Confidentiality clauses
6. Termination conditions

Please provide a structured draft of the MOU.
"""

# Generate draft using OpenAI's GPT model
response = client.chat.completions.create(
    model="gpt-3.5-turbo-16k",
    messages=[
        {"role": "system", "content": "You are a legal expert assistant specializing in drafting legal documents. Use the provided legal context to inform your draft."},
        {"role": "user", "content": detailed_prompt}
    ],
    max_tokens=1000,
    temperature=0.7
)

# Print the draft
print("Generated Draft:")
print(response.choices[0].message.content.strip())


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import fitz  # PyMuPDF or use pdfplumber if fitz fails
from langchain_community.llms import LlamaCpp

# Function to extract text from a PDF
def extract_text_from_pdf(pdf_path):
    pdf_document = fitz.open(pdf_path)  # Use fitz from PyMuPDF
    text = ""
    for page_num in range(pdf_document.page_count):
        page = pdf_document.load_page(page_num)
        text += page.get_text()
    return text

# Function to chunk text into manageable pieces
def chunk_text(text, max_length, tokenizer):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = [tokens[i:i + max_length] for i in range(0, len(tokens), max_length)]
    return [tokenizer.decode(chunk) for chunk in chunks]  # Decode tokens back to text

# Specify the path to your PDF file
pdf_path = "TYPE-SPECIAL LEAVE PETITION - JOHN DOE.pdf"

# Extract text from the PDF
text = extract_text_from_pdf(pdf_path)

# Load tokenizer and model from law-ai/InLegalBERT
tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
model = AutoModel.from_pretrained("law-ai/InLegalBERT").to('cuda')  # Move model to GPU

# Chunk the text
max_length = tokenizer.model_max_length  # Typically 512 for BERT-based models
chunks = chunk_text(text, max_length, tokenizer)

# Process each chunk
embeddings = []
for chunk in chunks:
    encoded_input = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True).to('cuda')
    with torch.no_grad():
        output = model(**encoded_input)
    last_hidden_state = output.last_hidden_state
    mean_state = torch.mean(last_hidden_state, dim=1).squeeze()
    embedding = mean_state.detach().cpu().numpy()
    embeddings.append(embedding)

# Aggregate embeddings
text_embedding = np.mean(embeddings, axis=0)

# Convert embedding to string
embedding_str = ','.join(map(str, text_embedding.flatten()))

# Load local model using LangChain
llm = LlamaCpp(
    model_path="/home/avinash/Documents/01Work/Models/gemma-2-27b-it-Q4_K_M.gguf",
    n_gpu_layers=-1,
    n_batch=8000,
    n_ctx=8000,
    f16_kv=True,
    verbose=True,
    max_tokens=25000
)

# Prepare prompt
detailed_prompt = f"""
Based on the legal context embedding: {embedding_str}

Draft a Memorandum of Understanding (MOU) between Party A and Party B. Consider the following points:
1. Purpose of the agreement
2. Scope of cooperation
3. Responsibilities of each party
4. Duration of the agreement
5. Confidentiality clauses
6. Termination conditions

Please provide a structured draft of the MOU.
"""

# Generate draft
result = llm.invoke(detailed_prompt)

# Print the draft
print("Generated Draft:")
print(result.strip())


In [2]:
import os
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import fitz  # PyMuPDF or use pdfplumber if fitz fails
from langchain_community.llms import LlamaCpp

# Set the CUDA_VISIBLE_DEVICES environment variable
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Change '0' to the appropriate GPU ID if necessary

# Function to extract text from a PDF
def extract_text_from_pdf(pdf_path):
    pdf_document = fitz.open(pdf_path)  # Use fitz from PyMuPDF
    text = ""
    for page_num in range(pdf_document.page_count):
        page = pdf_document.load_page(page_num)
        text += page.get_text()
    return text

# Function to chunk text into manageable pieces
def chunk_text(text, max_length, tokenizer):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = [tokens[i:i + max_length] for i in range(0, len(tokens), max_length)]
    return [tokenizer.decode(chunk) for chunk in chunks]  # Decode tokens back to text

# Specify the path to your PDF file
pdf_path = "TYPE-SPECIAL LEAVE PETITION - JOHN DOE.pdf"

# Extract text from the PDF
text = extract_text_from_pdf(pdf_path)

# Load tokenizer and model from law-ai/InLegalBERT
tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
model = AutoModel.from_pretrained("law-ai/InLegalBERT").to('cuda')  # Move model to GPU

# Chunk the text
max_length = tokenizer.model_max_length  # Typically 512 for BERT-based models
chunks = chunk_text(text, max_length, tokenizer)

# Process each chunk
embeddings = []
for chunk in chunks:
    encoded_input = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True).to('cuda')
    with torch.no_grad():
        output = model(**encoded_input)
    last_hidden_state = output.last_hidden_state.to('cuda')  # Ensure tensor is on GPU
    mean_state = torch.mean(last_hidden_state, dim=1).squeeze().to('cuda')  # Ensure tensor is on GPU
    embedding = mean_state.detach().cpu().numpy()  # Move to CPU for numpy conversion
    embeddings.append(embedding)

# Aggregate embeddings
text_embedding = np.mean(embeddings, axis=0)

# Convert embedding to string
embedding_str = ','.join(map(str, text_embedding.flatten()))

# Load local model using LangChain
llm = LlamaCpp(
    model_path="/home/avinash/Documents/01Work/Models/gemma-2-27b-it-Q4_K_M.gguf",
    n_gpu_layers=-1,
    n_batch=8000,
    n_ctx=8000,
    f16_kv=True,
    verbose=False,
    max_tokens=2048  # Adjust token limit as needed
)

# Function to generate long-form content
def generate_long_form_content(prompt, model, chunk_size=2048, total_length=10000):
    full_text = ""
    while len(full_text) < total_length:
        response = model.invoke(prompt)
        text_chunk = response.strip()
        full_text += text_chunk
        prompt = prompt + "\n" + text_chunk  # Append generated text to the prompt for continuity
        if len(text_chunk) < chunk_size:  # Stop if response is shorter than chunk size
            break
    return full_text

# Prepare prompt
detailed_prompt = f"""
Based on the legal context embedding: {embedding_str[:1000]}

Draft a special leave petition in a similar style. 

"""

# Generate long-form content
long_form_text = generate_long_form_content(detailed_prompt, llm)

# Print the draft
print("Generated Draft:")
print(long_form_text)


RuntimeError: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero.